Letterboxd data is saved super weird - this notebook was me working through .csv files and figuring out how to combine them into something decent to look at 


profile.csv - username, given name, FAVORITES***

diary.csv usefeul contents - 
    Per movie - name, release year, rating
    ratings.csv is all duplicate info
    watched.csv has more data BC not all ratings are logged movies 
    likes/films seems helpful, 
    
comments is worthless (i think), reviews and watchlist also seem unhelpful for now

based on data available per user(not much), the main stuff that can be used seems to be 
raw score(maybe something else like percentile w this?) and likes(4 favorites also exists but thats a small sample)
and using those to apply a weighted score for various criteria (i.e. 80s movies tend to score higher, Wes Anderson tends to score higher), creating a list of priorities
using an external database such as TMBD that will filter for those priorities


In [4]:
import pandas as pd
from pathlib import Path


def load_csv(name):
    candidates = [
        Path.cwd() / name,
        Path.cwd() / "my-data" / "raw" / name,
        Path.cwd().parent / "my-data" / "raw" / name,
        Path.cwd() / ".." / "my-data" / "raw" / name,
    ]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"Could not find {name}")


ratings_df = load_csv("ratings.csv")
watched_df = load_csv("watched.csv")

ratings_df.head(), watched_df.head()

(         Date                          Name  Year        Letterboxd URI  \
 0  2020-04-06        Avengers: Infinity War  2018  https://boxd.it/9vEe   
 1  2020-04-06             Avengers: Endgame  2019  https://boxd.it/9vE4   
 2  2020-04-06  Star Wars: The Force Awakens  2015  https://boxd.it/4vru   
 3  2020-04-06      Star Wars: The Last Jedi  2017  https://boxd.it/5xme   
 4  2020-04-06                 The Godfather  1972  https://boxd.it/2aNK   
 
    Rating  
 0     4.0  
 1     4.0  
 2     3.5  
 3     3.0  
 4     4.5  ,
          Date                               Name  Year        Letterboxd URI
 0  2020-04-06  Spider-Man: Into the Spider-Verse  2018  https://boxd.it/azpY
 1  2020-04-06             Avengers: Infinity War  2018  https://boxd.it/9vEe
 2  2020-04-06                    The Dark Knight  2008  https://boxd.it/2b0k
 3  2020-04-06                  Avengers: Endgame  2019  https://boxd.it/9vE4
 4  2020-04-06                      Black Panther  2018  https://boxd.it/

Taking watched.df as the skeleton, adding likes to it, keeping only any recent rewatch 

Plan - combine letterboxd dataframes, incl rating, liked, favorites(number rewatches?)
Eventually, add TMDB data containing avg rating, actors/directors, genres? 

In [16]:
import os
print(os.getcwd())

c:\Users\htkle\Documents\senior summer\movie-recs\notebooks


In [ ]:
"""
Combining my exports from letterboxd into an organized dataframe, with
watched.csv as the base and adding flags for liked and favorite films from 
liked/films.csv and profile.csv, respectively.


  - watched.csv        columns: Date, Name, Year, Letterboxd URI
  - likes/films.csv     columns: Date, Name, Year, Letterboxd URI
  - profile.csv         one row with a favorite films column(comma-separated URIs)

Note: diary.csv and reviews.csv use a different URI unique per viewing, 
but they are not currently being used in this algorithm """


# Notebook working dir is notebooks/, so resolve data relative to repo root.
root_dir = Path.cwd()
if not (root_dir / "my-data").exists():
    root_dir = root_dir.parent

DATA_DIR = root_dir / "my-data" / "raw"
WATCHED_CSV = DATA_DIR / "watched.csv"
LIKES_CSV = DATA_DIR / "likes" / "films.csv"
PROFILE_CSV = DATA_DIR / "profile.csv"

JOIN_KEY = "Letterboxd URI"
FAVORITE_COL = "Favorite Films" 


def inspect_columns():

    for label, path in [
        ("watched.csv", WATCHED_CSV),
        ("likes/films.csv", LIKES_CSV),
        ("profile.csv", PROFILE_CSV),
    ]:
        df = pd.read_csv(path)
        print(f"\n{label} columns:")
        print(list(df.columns))
        print(df.head(2))

def load_letterboxd_data():
    watched_df = pd.read_csv(WATCHED_CSV)
    likes_df = pd.read_csv(LIKES_CSV)
    profile_df = pd.read_csv(PROFILE_CSV)
    return watched_df, likes_df, profile_df



def combine_letterboxd_data(watched_df, likes_df, profile_df):
    df = watched_df.copy()

    # --- Liked flag ---
    if JOIN_KEY not in likes_df.columns:
        raise KeyError(
            f"'{JOIN_KEY}' not found in likes_df columns: {list(likes_df.columns)}. "
            "Run inspect_columns() to check the actual header names."
        )
    liked_uris = set(likes_df[JOIN_KEY].dropna())
    df["Liked"] = df[JOIN_KEY].isin(liked_uris)

    # --- Favorite flag ---
    if FAVORITE_COL not in profile_df.columns:
        raise KeyError(
            f"'{FAVORITE_COL}' not found in profile_df columns: "
            f"{list(profile_df.columns)}. Run inspect_columns() to check the actual "
            "header names."
        )

    raw = profile_df[FAVORITE_COL].iloc[0]
    favorite_uris = {uri.strip() for uri in raw.split(",")}
    df["Favorite"] = df[JOIN_KEY].isin(favorite_uris)

    return df

combined_df = combine_letterboxd_data(*load_letterboxd_data())
combined_df.head(10)

,Date,Name,Year,Letterboxd URI,Liked,Favorite
0,2020-04-06,Spider-Man: Into the Spider-Verse,2018,https://boxd.it/azpY,True,False
1,2020-04-06,Avengers: Infinity War,2018,https://boxd.it/9vEe,True,False
2,2020-04-06,The Dark Knight,2008,https://boxd.it/2b0k,True,False
3,2020-04-06,Avengers: Endgame,2019,https://boxd.it/9vE4,True,False
4,2020-04-06,Black Panther,2018,https://boxd.it/8MHs,True,False
5,2020-04-06,Guardians of the Galaxy,2014,https://boxd.it/3VH2,True,False
6,2020-04-06,The Shining,1980,https://boxd.it/29Nu,True,False
7,2020-04-06,Star Wars: The Force Awakens,2015,https://boxd.it/4vru,True,False
8,2020-04-06,Gone Girl,2014,https://boxd.it/6hQu,False,False
9,2020-04-06,Star Wars: The Last Jedi,2017,https://boxd.it/5xme,False,False


The above algorithm correctly returns every film a user has watched(for the current version of letterboxd's user downloaded data), regardless of it was logged in the diary, and adds two columns to classify if the film is listed as a top 4 favorite or if it has been 'liked' by the user. Now that I have code to sort through a given set of letterboxd data, it will be sorted into a reuseable pipeline for the project
note: issue in the individual files likely due to venv errors

The loading/sorting is added to the project, below is code to import those modules into the notebook to pick up any changes as needed(the above serves as a draft, but will likely not change much if at all for the time
)